# Expanded comparison: hybrid vs rerank20 vs rerank50

1. Upload **phase4_comparison_source.zip** to the root of My Drive.
2. Select an **A100 GPU** in Colab and Run all. Approve Drive mounting.
3. Bring the downloaded **phase4_comparison_results.zip** back for analysis.

Use this notebook and ZIP together. Source is loaded from Drive; no encoded archive cell.
Packages install with uv. Three pinned repos, 30 questions each: **90 total**, including
75 additions and the original 15. Types: 36 ambiguous, 18 multi-block, 18 test, 12 implementation,
6 example. Ambiguous queries omit test/example/symbol cues; explicit target guards stay separate.

Each policy runs independently through production retrieval, with fixed weights and no tuning.
Rerank20 scores 20 candidates; rerank50 scores 50. All use the same first-stage depth of 50
per signal and return 10. Three rotated calls per policy/question: **810 measured calls**.
Warm timings include query encoding and reranking. Initialization/warmup is separate.
Scoring uses the pre-chosen repetition zero; repetitions are not extra questions.

Reports compare MRR, complete accepted proof, paired 50-minus-20 intervals, and per-repo
p50/p95. Inspect each type and the added75/retained15 views, not just the pooled mean.
Labels, protocol, full candidates, scores, source ZIP and revisions travel with the results.

Source-authored diagnostic sample; independent review pending. More questions improve coverage
but do not guarantee significance or independence. Six example cases remain a small guard set.
This does not rerun the AST/character comparison. **A100 is experiments only; deployment is
undecided. Reranking stays opt-in.** See the bundled QUESTIONS.md for label review.

In [ ]:
SOURCE_ZIP = "/content/drive/MyDrive/phase4_comparison_source.zip"

In [ ]:
# Load the separate source ZIP that you uploaded to Google Drive.
import hashlib, pathlib, zipfile
from google.colab import drive

drive.mount("/content/drive")
source_zip = pathlib.Path(SOURCE_ZIP)
if not source_zip.is_file():
    raise FileNotFoundError(
        f"Upload phase4_comparison_source.zip to My Drive, or update SOURCE_ZIP. Missing: {source_zip}"
    )
expected_sha256 = "9b00dc799d4f535f2f596815e6ffadd9b12f088cdea8f00b43266497f1334f15"
if hashlib.sha256(source_zip.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError(
        "Source ZIP does not match this notebook. Download the matching ZIP and notebook again."
    )
PAYLOAD = pathlib.Path("/content/phase4_comparison_payload")
with zipfile.ZipFile(source_zip) as package:
    package.extractall(PAYLOAD)
print("Loaded project code and question datasets from", source_zip.name)

In [ ]:
import importlib.util, subprocess, sys
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], check=True)
if importlib.util.find_spec('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--system',
                '--python', sys.executable, '-q', 'chromadb==1.5.9', 'fastapi==0.162.0', 'gitpython==3.2.1', 'langchain-chroma==1.1.0', 'langchain-core==1.6.13', 'langchain-text-splitters==1.1.3', 'langgraph==1.2.19', 'pydantic==2.14.1', 'pydantic-settings==2.16.1', 'rank-bm25==0.2.2', 'rich==15.0.0', 'sentence-transformers==5.7.0', 'transformers==4.57.6', 'tree-sitter==0.26.0', 'tree-sitter-language-pack==1.22.1', 'typer==0.27.3', 'uvicorn==0.54.0', 'astchunk==0.1.0'], check=True)

In [ ]:
import json, os, pathlib, shutil, subprocess, sys
REPOS = pathlib.Path('/content/comparison_repos')
RESULTS = pathlib.Path('/content/comparison_results')
LABELS = PAYLOAD / 'eval/datasets/phase4_comparison_v3'
if RESULTS.exists():
    raise FileExistsError('Results already exist. Download them before starting a new runtime.')
REPOS.mkdir(exist_ok=True)
RESULTS.mkdir()
shutil.copytree(LABELS, RESULTS / 'labels')
shutil.copyfile(source_zip, RESULTS / 'phase4_comparison_source.zip')
env = {**os.environ, 'PYTHONPATH': os.pathsep.join([str(PAYLOAD), str(PAYLOAD / 'src')]),
       'TOKENIZERS_PARALLELISM': 'false'}
manifest = json.loads((LABELS / 'manifest.json').read_text())
for entry in manifest['datasets']:
    repo = entry['repo']
    checkout = REPOS / repo.replace('/', '__')
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                        'https://github.com/' + repo, str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach',
                    entry['commit_sha']], check=True)
    dataset = LABELS / entry['file']
    if hashlib.sha256(dataset.read_bytes()).hexdigest() != entry['sha256']:
        raise ValueError('Dataset checksum mismatch')
    subprocess.run([sys.executable, str(PAYLOAD / 'eval/run_ambiguous_eval.py'),
                    '--dataset', str(dataset), '--checkout', str(checkout),
                    '--output', str(RESULTS / repo.replace('/', '__')),
                    '--protocol', str(LABELS / 'protocol.json'),
                    '--mode', 'full', '--device', 'cuda', '--repetitions', '3'],
                    env=env, check=True)
subprocess.run([sys.executable, str(PAYLOAD / 'eval/summarize_comparison.py'),
                '--results', str(RESULTS), '--manifest', str(RESULTS / 'labels/manifest.json')],
                env=env, check=True)
print((RESULTS / 'aggregate/summary.md').read_text())

In [ ]:
from google.colab import files
archive = pathlib.Path('/content/phase4_comparison_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS.rglob('*')):
        if path.is_file() and 'isolated_index' not in path.parts:
            bundle.write(path, path.relative_to(RESULTS))
files.download(str(archive))